In [1]:
# First you need to initialize ollama serve like this:
#export OLLAMA_HOST=127.0.0.1:11500 <-- you need a different port from the default one because your mac keeps the default one busy
#OLLAMA_NUM_PARALLEL=3 ollama serve

import os
# os.environ["OLLAMA_HOST"] = "127.0.0.1:11500"

In [ ]:
from ollama import chat, ChatResponse
from typing import List, Dict, Tuple
from collections import Counter


TOPIC_LABELS = [
    "national identity",
    "migration & xenophobia",
    "antisemitism",
    "islamophobia",
    "conspiracy narratives",
    "street action / militancy",
    "ideology / doctrine",
    "electoral politics",
    "violent rhetoric / incitement"
]

def prompt_llm(message_text: str, llm_model: str = "your_model") -> Tuple[List[str], List[Dict]]:
    system_prompt = f"""You are a strict classifier. For the provided social media message,
identify the smallest set of the most prominent topic(s) from this list (choose none if none apply):
{TOPIC_LABELS}

Also extract ALL explicit or implicit location mentions (countries, regions, cities, neighbourhoods, landmarks, geopolitical entities).
Count exact string occurrences case-insensitively and merge equivalent forms (e.g., "UK" and "United Kingdom" → "United Kingdom").
Sort locations by count descending, then alphabetically.

You MUST respond by CALLING the provided tool with JSON conforming to the tool schema.
Do NOT produce any other plain-text output."""

    extract_tool = {
        "type": "function",
        "function": {
            "name": "extract_topics_locations",
            "description": "Return topics (subset of canonical labels) and locations with counts for the given message.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topics": {
                        "type": "array",
                        "items": {"type": "string", "enum": TOPIC_LABELS}
                    },
                    "locations_ranked": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "location": {"type": "string"},
                                "count": {"type": "integer", "minimum": 1}
                            },
                            "required": ["location", "count"]
                        }
                    }
                },
                "required": ["topics", "locations_ranked"]
            }
        }
    }

    response: ChatResponse = chat(
        model=llm_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": message_text}
        ],
        tools=[extract_tool],
    )

    tool_calls = response.message.tool_calls
    if not tool_calls:
        response_text = response.message.content.strip() if response.message.content else "(no content)"
        raise ValueError(f"Model did not call the tool. Output was:\n{response_text}")

    call = tool_calls[0]
    args = call.function.arguments

    if not isinstance(args, dict):
        raise ValueError("Tool arguments were not a dict/JSON object.")

    topics = args.get("topics")
    locations_ranked = args.get("locations_ranked")

    if topics is None or locations_ranked is None:
        raise ValueError(f"Tool call missing required fields. Got: {args}")

    if not isinstance(topics, list) or any(t not in TOPIC_LABELS for t in topics):
        raise ValueError(f"Invalid topics returned: {topics}")

    if not isinstance(locations_ranked, list):
        raise ValueError("locations_ranked must be a list.")

    for item in locations_ranked:
        if not isinstance(item, dict) or "location" not in item or "count" not in item:
            raise ValueError(f"Invalid location entry: {item}")

    return topics, locations_ranked

In [3]:
model = "qwen3:1.7b"
workers = 3
sample_size = 1000

In [4]:
import pandas as pd

data = pd.read_csv('data/jungenationalisten/message_nodes.csv')

data = data[data['text'].notna()].copy()

sample = data.sample(n=sample_size, random_state=42)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time


def tag_single_text(text, model_name="your_model"):
    try:
        topics, locations = prompt_llm(text, llm_model=model_name)
        return {"topics": topics, "locations_ranked": locations, "error": None}
    except Exception as e:
        return {"topics": None, "locations_ranked": None, "error": str(e)}

sample["topics"] = None
sample["locations_ranked"] = None
sample["tagging_error"] = None

texts_with_indices = [(idx, text) for idx, text in zip(sample.index, sample["text"])]

results = []

start_time = time.time()

with ThreadPoolExecutor(max_workers=workers) as executor:
    futures = {executor.submit(tag_single_text, text, model): idx for idx, text in texts_with_indices}

    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing texts"):
        idx = futures[future]
        results.append((idx, future.result()))

end_time = time.time()
total_time = end_time - start_time

for idx, result in results:
    sample.at[idx, "topics"] = result["topics"]
    sample.at[idx, "locations_ranked"] = result["locations_ranked"]
    sample.at[idx, "tagging_error"] = result["error"]

sample.to_csv("data/message_nodes_tagged.csv", index=False)

print(f"\n{'='*60}")
print(f"Total time: {total_time:.2f} seconds")
print(f"Average time per prompt: {total_time / len(texts_with_indices):.2f} seconds")
print(f"Prompts per second: {len(texts_with_indices) / total_time:.2f}")
print(f"{'='*60}")


Processing texts:  19%|█▉        | 190/1000 [23:31<1:40:16,  7.43s/it]


In [ ]:
sample.head()

,id,chat,message_id,date,url,views,reaction_count,reaction_breakdown,text,sender_id,topics,locations_ranked,tagging_error
20140,Stimme_der_Wahrheit:3393,Stimme_der_Wahrheit,3393,2025-11-18T18:18:03+00:00,https://t.me/Stimme_der_Wahrheit/3393,939,10,"{""🔥"": 9, ""🫡"": 1}",18. November 1919:\nPaul von Hindenburg gibt m...,-1002257143821,None,None,model 'qwen3:1.7b' not found (status code: 404)
7897,gegenstrom:131,gegenstrom,131,2020-12-23T05:53:18+00:00,https://t.me/gegenstrom/131,648,0,{},"Rezension: ""Kampf um Europa""\n\nNachfolgend ve...",-1001227660512,None,None,model 'qwen3:1.7b' not found (status code: 404)
8912,sicherheitshinweise:1190,sicherheitshinweise,1190,2023-04-05T17:40:38+00:00,https://t.me/sicherheitshinweise/1190,26224,52,"{""🔥"": 28, ""😢"": 13, ""👍"": 11}",Wie wurde Rob Rundo gefunden?\n\nWie die meist...,-1001338732850,None,None,model 'qwen3:1.7b' not found (status code: 404)
1194,schmidtkeswelt:2267,schmidtkeswelt,2267,2025-06-11T12:01:33+00:00,https://t.me/schmidtkeswelt/2267,1068,17,"{""🔥"": 11, ""🤣"": 5, ""❤"": 1}",🔥 WENDE IM COMPACT-VERFAHREN? Eine in der Mitt...,-1001441697781,None,None,model 'qwen3:1.7b' not found (status code: 404)
27870,nsfronta:586,nsfronta,586,2025-11-11T11:25:12+00:00,https://t.me/nsfronta/586,923,42,"{""❤"": 33, ""⚡"": 8, ""👎"": 1}",Dnes vzdáváme poctu všem veteránům.\nDíky za o...,-1001499347321,None,None,model 'qwen3:1.7b' not found (status code: 404)
